# TT-rank / unfolding: 一般d階テンソルでの確認

前Notebookの3階TT-SVDを一般 $d$ 階へ拡張し、TT coreのbond dimensionと元テンソルのcut unfolding rankを照合する。

## ゴール
- cut unfoldingを実装する
- 各cutのrankを確認する
- 打ち切りなしTT-SVDを一般化する
- coreから読めるTT-rankとunfolding rankを比較する
- 完全再構成を確認する

> `torch.linalg.matrix_rank` はしきい値に基づく数値rankを返す。


## 1. 実験対象

$X\in\mathbb{R}^{2\times3\times2\times2}$ を使う。


In [20]:
import math
import torch

torch.set_default_dtype(torch.float64)
torch.manual_seed(1)

# TODO: shape=(2, 3, 2, 2) の X を作る
X = torch.randn(2, 3, 2,2)

In [21]:
from nn_compression.tensor.validation import (
    validate_mode_index,
    validate_positive_shape,
    validate_tensor_shape,
)
from nn_compression.compression import (
    truncated_svd,
)
from nn_compression.metrics import relative_frobenius_error


## 2. cut unfolding

切断 $1{:}k \mid k{+}1{:}d$ に対応する行列を作る。

4階ならshapeは
- $k=1$: $(n_1,\ n_2n_3n_4)$
- $k=2$: $(n_1n_2,\ n_3n_4)$
- $k=3$: $(n_1n_2n_3,\ n_4)$


In [22]:
def tt_unfold(X: torch.Tensor, k: int) -> torch.Tensor:
    """cut 1:k | k+1:d に対応するunfolding。"""
    """TTのcut 1:k | k+1:d に対応する2次元unfoldingを返す。"""

    # ndim >= 2 かつ各 dimension が正の整数であることを確認する
    validate_tensor_shape(X)

    # kが整数でかつ、0<=k<=X.ndimであることの確認
    validate_mode_index(k, X.ndim)
    
    shape = X.shape
    left_dim = math.prod(shape[:k])
    right_dim = math.prod(shape[k:])

    return X.reshape(left_dim, right_dim)




## 3. unfolding rank

\[
r_k=
\operatorname{rank}
\left(
X_{(i_1\cdots i_k),(i_{k+1}\cdots i_d)}
\right)
\]

各cutのshapeとrankを求め、`exact_ranks` に保存する。


In [23]:
exact_ranks = []

for k in range(1, X.ndim):
    unfolded = tt_unfold(X, k)
    r = torch.linalg.matrix_rank(unfolded)
    exact_ranks.append({
        "cut": k,
        "shape": tuple(unfolded.shape),
        "rank": r,
    })


## 4. 一般d階のTT-SVD

```python
tt_svd_exact(X) -> list[torch.Tensor]
```

各coreは

\[
G^{(k)}\in\mathbb{R}^{r_{k-1}\times n_k\times r_k}
\]

のshapeを持つ。打ち切りなしでは、各段階の数値rankを残す。


In [24]:
def tt_svd_exact(X: torch.Tensor) -> list[torch.Tensor]:
    """任意の d 階テンソルを、数値 rank を保ったまま TT core 列へ分解する。"""
    validate_tensor_shape(X)

    shape = X.shape
    d = X.ndim
    cores: list[torch.Tensor] = []

    # remainder: 未分解の残り。shape は (r_left, n_mode, n_{mode+1}, ..., n_d)
    remainder = X
    r_left = 1  # 左端の bond dimension（最初は 1）

    # d 階テンソルなら SVD は d-1 回。各 core は G^(k) ∈ R^{r_{k-1} × n_k × r_k}
    for mode in range(d - 1):
        n_mode = shape[mode]
        # 切断 (r_left·n_mode) | (残り) に対応する 2 次元行列
        mat = remainder.reshape(r_left * n_mode, -1)

        # 打ち切りなし: unfolding の数値 rank を bond dimension として残す
        r = int(torch.linalg.matrix_rank(mat).item())
        U, S, Vh = truncated_svd(mat, r)
        cores.append(U.reshape(r_left, n_mode, r))

        # Σ V^T を次の remainder へ。物理モード n_{mode+1}, ..., n_d を復元
        remainder = (torch.diag(S) @ Vh).reshape(r, *shape[mode + 1 :])
        r_left = r

    # 最後の core。右端 bond は TT 形式どおり 1
    cores.append(remainder.reshape(r_left, shape[-1], 1))
    return cores


## 5. TTから再構成

```python
tt_reconstruct(cores)
```

bondを縮約し、元のdense tensorを再構成する。


In [25]:
def tt_reconstruct(cores: list[torch.Tensor]) -> torch.Tensor:
    """TT core 列を左から bond 縮約し、dense テンソルを再構成する。"""
    result = cores[0]
    for core in cores[1:]:
        # 左 core の右 bond（最終軸）と右 core の左 bond（先頭軸）を縮約
        result = torch.tensordot(result, core, dims=([-1], [0]))
    return result.squeeze(0).squeeze(-1)


## 6. TT-rankの照合

- core shapeを表示する
- `cores[:-1]` の右bond dimensionからTT-rank列を取り出す
- `exact_ranks` と比較する


In [26]:
cores = tt_svd_exact(X)

for i, core in enumerate(cores, start=1):
    print(f"G{i}.shape = {tuple(core.shape)}")

core_ranks = [core.shape[2] for core in cores[:-1]]
unfold_ranks = [int(item["rank"].item()) for item in exact_ranks]
print("core_ranks:", core_ranks)
print("unfold_ranks:", unfold_ranks)
print("match:", core_ranks == unfold_ranks)


G1.shape = (1, 2, 2)
G2.shape = (2, 3, 4)
G3.shape = (4, 2, 2)
G4.shape = (2, 2, 1)
core_ranks: [2, 4, 2]
unfold_ranks: [2, 4, 2]
match: True


## 7. 完全再構成と保存量

- relative Frobenius error
- dense要素数
- TT core総要素数

を比較する。


In [28]:
X_hat = tt_reconstruct(cores)
rel_error = relative_frobenius_error(X, X_hat)

dense_params = X.numel()
tt_params = sum(core.numel() for core in cores)

print(f"X_hat.shape = {tuple(X_hat.shape)}")
print(f"relative Frobenius error = {rel_error.item():.3e}")
print(f"dense params = {dense_params}")
print(f"TT params = {tt_params}")


X_hat.shape = (2, 3, 2, 2)
relative Frobenius error = 6.414e-16
dense params = 24
TT params = 48
